<a href="https://colab.research.google.com/github/ypsitau/google-colab/blob/main/create_recognizer_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Create Recognizer Model from EMNIST

Running the script creates the following TfLite model files. They will be automatically downloaded to your remote PC after the creation.

|File|Note|
|----|----|
|`Recognizer-EMNIST-mnist-binary.tflite`|Digits (0-9)|
|`Recognizer-EMNIST-letters-binary.tflite`|Upper letters (A-Z)|
|`Recognizer-EMNIST-bymerge-binary.tflite`|Digits (0-9), upper letters (A-Z), and lower letters (a-z)|

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import google.colab

def generate_model(config_name: str, epochs: int, binary_data: bool) -> str:
    label_offset = 1 if config_name == 'letters' else 0
    (dataset_train, dataset_test), dataset_info = tfds.load( # tf.data.Dataset
        f'emnist/{config_name}',
        split=['train', 'test'],
        as_supervised=True, # each data element is a tuple (image, label)
        with_info=True,
    )
    num_classes = dataset_info.features['label'].num_classes - label_offset
    #---------------------------------------------------------------------------
    print(f"{config_name}: {num_classes} classes")
    def preprocess(image: tf.Tensor, label: tf.Tensor) -> tuple[tf.Tensor, tf.Tensor]:
        image = tf.transpose(image, perm=[1, 0, 2])
        image = tf.cast(image > 127, tf.float32) if binary_data else tf.cast(image, tf.float32) / 255.0
        label = label - label_offset
        return image, label
    dataset_train = dataset_train.map(preprocess).cache().shuffle(10000).batch(64).prefetch(tf.data.AUTOTUNE)
    dataset_test = dataset_test.map(preprocess).batch(64).cache().prefetch(tf.data.AUTOTUNE)
    model = tf.keras.models.Sequential([
        tf.keras.layers.Conv2D(16, (3, 3), activation='relu', input_shape=(28, 28, 1)),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    model.fit(dataset_train, epochs=epochs, validation_data=dataset_test)
    #---------------------------------------------------------------------------
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    def representative_dataset_generator():
        for image, label in dataset_test.take(100):
            yield [image]
    converter.representative_dataset = representative_dataset_generator
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8
    model_tflite = converter.convert()
    filename = f"Recognizer-EMNIST-{config_name}{'-binary' if binary_data else ''}.tflite"
    with open(filename, "wb") as f:
        f.write(model_tflite)
    return filename

filenames = []
filenames.append(generate_model('mnist', 5, True))     # 0-9
filenames.append(generate_model('letters', 10, True))  # A-Z, 0-9
filenames.append(generate_model('bymerge', 10, True))  # 0-9, A-Z, a, b, d, e, f, g, h, n, q, r, t
for filename in filenames:
    google.colab.files.download(filename)
